# Utility usage forecast (LSTM, next 30 days)

Trains one simple univariate LSTM per meter and writes a 30-day forecast CSV + metadata JSON
for each, into this same `public/forecasts/` folder.

**Sources** (`public/energy/`): `Total_power.csv`, `Total_gas.csv`, `water.csv`, `battery.csv` — all 15-minute
cumulative meter readings. Each is converted to a **daily delta** (last reading of day minus last reading
of the previous day, clipped at 0) before forecasting.

**Method** mirrors the existing JS pipeline (`src/simulation/energyForecastLstm.js` /
`npm run forecast:gen`): single-layer LSTM, min-max normalization, chronological 80/10/10
train/val/test split, autoregressive multi-step forecast.

**Output format**: `date,series,value` CSV per meter (`series` is `history` or `forecast`) plus a
`_meta.json` with metrics — same shape as `public/energy/lstm_import_forecast.csv`, using a generic
`value` column instead of `kwh` since these cover different units. Not yet wired into the app's
Forecast tab — this notebook only produces the files.

Run all cells top to bottom. Takes a few minutes (4 models, ~28 epochs each).

In [ ]:
import json
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

np.random.seed(7)
tf.random.set_seed(7)

## Config

In [ ]:
NOTEBOOK_DIR = Path.cwd()
SOURCE_DIR = NOTEBOOK_DIR.parent / "energy"
OUTPUT_DIR = NOTEBOOK_DIR

LOOKBACK = 14
FORECAST_DAYS = 30
LSTM_UNITS = 16
EPOCHS = 28
BATCH_SIZE = 16
MIN_DAILY_POINTS = LOOKBACK + 40

SERIES = [
    {
        "key": "power",
        "source_csv": SOURCE_DIR / "Total_power.csv",
        "value_cols": ["Import T1 kWh", "Import T2 kWh"],
        "label": "Daily grid import",
        "unit": "kWh",
        "scale": 1.0,
        "output_csv": OUTPUT_DIR / "power_forecast.csv",
        "output_meta": OUTPUT_DIR / "power_forecast_meta.json",
    },
    {
        "key": "gas",
        "source_csv": SOURCE_DIR / "Total_gas.csv",
        "value_cols": ["Total gas used"],
        "label": "Daily gas usage",
        "unit": "m3",
        "scale": 1.0,
        "output_csv": OUTPUT_DIR / "gas_forecast.csv",
        "output_meta": OUTPUT_DIR / "gas_forecast_meta.json",
    },
    {
        "key": "water",
        "source_csv": SOURCE_DIR / "water.csv",
        "value_cols": ["water usage dl"],
        "label": "Daily water usage",
        "unit": "L",
        "scale": 0.1,  # source is deciliters -> liters
        "output_csv": OUTPUT_DIR / "water_forecast.csv",
        "output_meta": OUTPUT_DIR / "water_forecast_meta.json",
    },
    {
        "key": "battery",
        "source_csv": SOURCE_DIR / "battery.csv",
        "value_cols": ["Import kWh"],
        "label": "Daily battery import",
        "unit": "kWh",
        "scale": 1.0,
        "output_csv": OUTPUT_DIR / "battery_forecast.csv",
        "output_meta": OUTPUT_DIR / "battery_forecast_meta.json",
    },
]

## Helpers

`load_daily_series` turns 15-minute cumulative readings into a daily delta series
(last reading of each day, differenced, clipped at 0, unit-scaled).

In [ ]:
def load_daily_series(cfg):
    df = pd.read_csv(cfg["source_csv"])
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["time"])
    df["cumulative"] = df[cfg["value_cols"]].sum(axis=1, min_count=1)
    df = df.dropna(subset=["cumulative"])
    df["date"] = df["time"].dt.date

    daily_last = df.groupby("date")["cumulative"].last().sort_index()
    daily_delta = daily_last.diff().dropna()
    daily_delta = daily_delta.clip(lower=0) * cfg["scale"]

    return pd.DataFrame({
        "date": [d.isoformat() for d in daily_delta.index],
        "value": daily_delta.to_numpy().round(3),
    })

`train_and_forecast` builds sliding windows, trains a 1-layer LSTM, and forecasts
`FORECAST_DAYS` steps ahead by feeding each prediction back in as the next input (autoregressive).

In [ ]:
def build_windows(values, lookback):
    xs, ys = [], []
    for i in range(lookback, len(values)):
        xs.append(values[i - lookback:i])
        ys.append(values[i])
    return np.array(xs)[..., np.newaxis], np.array(ys)


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))) if len(y_true) else None


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))) if len(y_true) else None


def train_and_forecast(daily_df, cfg):
    values = daily_df["value"].to_numpy(dtype="float32")
    dates = daily_df["date"].tolist()

    if len(values) < MIN_DAILY_POINTS:
        raise ValueError(
            f"{cfg['key']}: need at least {MIN_DAILY_POINTS} daily points, have {len(values)}"
        )

    v_min, v_max = float(values.min()), float(values.max())
    span = (v_max - v_min) or 1.0
    norm = (values - v_min) / span

    xs, ys = build_windows(norm, LOOKBACK)
    n = len(xs)
    n_train = max(1, int(n * 0.8))
    n_val = max(1, int(n * 0.1))
    i_val_end = n_train + n_val

    x_train, y_train = xs[:n_train], ys[:n_train]
    x_val, y_val = xs[n_train:i_val_end], ys[n_train:i_val_end]
    x_test, y_test = xs[i_val_end:], ys[i_val_end:]

    if len(x_test) < 1:
        raise ValueError(f"{cfg['key']}: not enough windows for an 80/10/10 split")

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(LOOKBACK, 1)),
        tf.keras.layers.LSTM(LSTM_UNITS),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss="mse")

    val_data = (x_val, y_val) if len(x_val) else None
    model.fit(
        x_train, y_train,
        validation_data=val_data,
        epochs=EPOCHS,
        batch_size=min(BATCH_SIZE, max(1, len(x_train))),
        shuffle=False,
        verbose=0,
    )

    def denorm(arr):
        return arr * span + v_min

    train_pred = denorm(model.predict(x_train, verbose=0).flatten()) if len(x_train) else np.array([])
    val_pred = denorm(model.predict(x_val, verbose=0).flatten()) if len(x_val) else np.array([])
    test_pred = denorm(model.predict(x_test, verbose=0).flatten()) if len(x_test) else np.array([])

    train_true = denorm(y_train)
    val_true = denorm(y_val) if len(y_val) else np.array([])
    test_true = denorm(y_test) if len(y_test) else np.array([])

    metrics = {
        "trainMae": mae(train_true, train_pred),
        "valMae": mae(val_true, val_pred) if len(val_true) else None,
        "testMae": mae(test_true, test_pred),
        "testRmse": rmse(test_true, test_pred),
    }

    window = norm[-LOOKBACK:].tolist()
    last_date = datetime.fromisoformat(dates[-1])
    forecast = []
    for step in range(FORECAST_DAYS):
        x_in = np.array(window[-LOOKBACK:], dtype="float32").reshape(1, LOOKBACK, 1)
        next_norm = float(model.predict(x_in, verbose=0)[0, 0])
        next_val = max(0.0, next_norm * span + v_min)
        next_date = (last_date + timedelta(days=step + 1)).date().isoformat()
        forecast.append({"date": next_date, "value": round(next_val, 3)})
        window.append(next_norm)

    result = {
        "history": daily_df.tail(60).to_dict("records"),
        "forecast": forecast,
        "lookback": LOOKBACK,
        "forecastDays": FORECAST_DAYS,
        "split": {
            "trainPct": 80, "valPct": 10, "testPct": 10,
            "nWindows": n, "nTrain": len(x_train), "nVal": len(x_val), "nTest": len(x_test),
        },
        "metrics": metrics,
        "forecastTotal": round(float(sum(d["value"] for d in forecast)), 3),
    }
    return result, model

`write_outputs` writes the `date,series,value` CSV and the `_meta.json` alongside it.

In [ ]:
def write_outputs(cfg, result):
    lines = ["date,series,value"]
    for row in result["history"]:
        lines.append(f"{row['date']},history,{row['value']}")
    for row in result["forecast"]:
        lines.append(f"{row['date']},forecast,{row['value']}")
    cfg["output_csv"].write_text("\n".join(lines) + "\n", encoding="utf-8")

    meta = {
        "generatedAt": datetime.now().isoformat(timespec="seconds"),
        "target": cfg["key"],
        "label": cfg["label"],
        "unit": cfg["unit"],
        "model": "lstm",
        "lookback": result["lookback"],
        "forecastDays": result["forecastDays"],
        "split": result["split"],
        "metrics": result["metrics"],
        "forecastTotal": result["forecastTotal"],
        "sourceFile": cfg["source_csv"].name,
        "note": "Chronological 80/10/10 split. Regenerate by re-running this notebook.",
    }
    cfg["output_meta"].write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

## Run: train + forecast + write, for every meter

In [ ]:
results = {}
for cfg in SERIES:
    print(f"=== {cfg['key']} ({cfg['label']}) ===")
    daily = load_daily_series(cfg)
    print(f"  daily points: {len(daily)}")

    result, model = train_and_forecast(daily, cfg)
    write_outputs(cfg, result)
    results[cfg["key"]] = result

    m = result["metrics"]
    print(f"  test MAE: {m['testMae']:.3f} {cfg['unit']}  ·  test RMSE: {m['testRmse']:.3f} {cfg['unit']}")
    print(f"  next {FORECAST_DAYS}d total: {result['forecastTotal']:.1f} {cfg['unit']}")
    print(f"  wrote {cfg['output_csv'].name} and {cfg['output_meta'].name}")
    print()

    del model
    tf.keras.backend.clear_session()

## Sanity-check plots

In [ ]:
fig, axes = plt.subplots(len(SERIES), 1, figsize=(10, 3.5 * len(SERIES)))
for ax, cfg in zip(axes, SERIES):
    r = results[cfg["key"]]
    hist_dates = [d["date"] for d in r["history"]]
    hist_vals = [d["value"] for d in r["history"]]
    fc_dates = [d["date"] for d in r["forecast"]]
    fc_vals = [d["value"] for d in r["forecast"]]

    ax.plot(hist_dates, hist_vals, label="history (last 60d)")
    ax.plot(fc_dates, fc_vals, label=f"forecast (+{FORECAST_DAYS}d)", linestyle="--")
    ax.set_title(f"{cfg['label']} ({cfg['unit']})")
    ax.set_xticks(ax.get_xticks()[::7])
    ax.tick_params(axis="x", rotation=45)
    ax.legend()

plt.tight_layout()
plt.show()

## Next steps

Re-run this notebook whenever the source CSVs in `public/energy/` are refreshed — it overwrites
`power_forecast.csv`, `gas_forecast.csv`, `water_forecast.csv`, `battery_forecast.csv` and their
`_meta.json` files in this folder.

To wire these into the app's Forecast tab, extend `src/data/energyForecastService.js` to also
fetch from `/forecasts/<key>_forecast.csv` (parsing the `value` column) and add chart/KPI panels
in `src/panels/energyCharts.js` alongside the existing grid-import forecast.